# 05.7 Type Conversion and Casting

Python is **strongly typed**: it will not silently convert a string to a number
to make your expression work. You must ask. This notebook covers how to ask, and
what happens when the conversion cannot succeed.

## Theory

### Implicit vs explicit

Python does perform **some** automatic conversion — but only within the numeric
tower, where no information is lost:

```python
1 + 2.0        # 3.0   int promoted to float, safe
1 + Decimal(2) # works, int promoted to Decimal
"1" + 2        # TypeError - Python refuses to guess
```

The rule is that Python widens along `int -> float -> complex`, because every
integer is a valid float. It never converts between text and numbers, because
that would require guessing your intent.

### The conversion functions

| Function | Produces | Notes |
|---|---|---|
| `int(x)` | Integer | **Truncates** floats towards zero |
| `float(x)` | Float | Accepts `"inf"`, `"nan"` |
| `str(x)` | String | Works on anything |
| `bool(x)` | Boolean | Follows truthiness (05.5) |
| `complex(x)` | Complex | From a number or a string |

### `int()` truncates, it does not round

```python
int(3.9)    # 3, not 4
int(-3.9)   # -3, not -4  (towards zero, unlike //)
```

That is a different rule from `//`, which rounds towards negative infinity. It
catches people regularly.

### Parsing user input

Every value from `input()`, a file, or a web request is **text**. Converting it
is where invalid data must be caught — which means `try`/`except ValueError`
around every conversion of untrusted input.

In [ ]:
# Python promotes automatically WITHIN the numeric tower.
from decimal import Decimal
from fractions import Fraction

print("Automatic promotion (no data lost):")
print("   1 + 2.0        ->", 1 + 2.0, type(1 + 2.0).__name__)
print("   1 + 2j         ->", 1 + 2j, type(1 + 2j).__name__)
print("   1.5 + 2j       ->", 1.5 + 2j)
print("   True + 1       ->", True + 1, "<- bool is an int")
print("   Fraction(1,2) + 1 ->", Fraction(1, 2) + 1)

# But NOT between text and numbers.
print("")
print("No automatic conversion between text and numbers:")
try:
    "1" + 2
except TypeError as error:
    print("   '1' + 2 ->", error)

try:
    Decimal("1.5") + 1.5
except TypeError as error:
    print("   Decimal + float ->", error)

print("")
print("The second refusal is deliberate: mixing exact and approximate")
print("arithmetic silently would defeat the point of Decimal.")

In [ ]:
# int() TRUNCATES towards zero - it does not round.
print("int() truncates towards zero:")
print("")
print("   Value    int()   round()   //1      math.floor")
print("   " + "-" * 50)

import math
for value in [3.9, 3.1, -3.9, -3.1]:
    print(f"   {value:<8} {int(value):<7} {round(value):<9} "
          f"{value // 1:<8} {math.floor(value)}")

print("")
print("Note int(-3.9) is -3 but math.floor(-3.9) is -4.")
print("int() goes towards ZERO; floor goes towards NEGATIVE INFINITY.")

# int() on a string does NOT accept decimals.
print("")
print("int() on strings is strict:")
print("   int('42')    ->", int("42"))
print("   int('  42 ') ->", int("  42 "), "<- whitespace is fine")
try:
    int("42.7")
except ValueError as error:
    print("   int('42.7')  ->", error)
print("   int(float('42.7')) ->", int(float("42.7")), "<- convert twice")

## Converting strings safely

Anything from outside your program is untrusted text. Conversion is where you
validate it.

In [ ]:
# What float() and int() accept and reject.
candidates = ["42", "42.7", "  42  ", "4_2", "1e3", "inf", "nan",
              "", "abc", "42abc", "0x1F", "٤٢"]

print("Input        int()          float()")
print("-" * 52)

for text in candidates:
    # Try int().
    try:
        int_result = repr(int(text))
    except ValueError:
        int_result = "ValueError"

    # Try float().
    try:
        float_result = repr(float(text))
    except ValueError:
        float_result = "ValueError"

    print(f"{repr(text):<12} {int_result:<14} {float_result}")

print("")
print("Notes:")
print("   '4_2'  - underscores are allowed, matching literal syntax")
print("   '1e3'  - scientific notation is float-only")
print("   '0x1F' - needs int('0x1F', 16)")
print("   arabic-indic digits are accepted by int() - Unicode aware")

In [ ]:
# The safe conversion pattern for untrusted input.
def parse_int(text, default=None):
    """Convert text to an int, returning a default when it fails."""
    try:
        return int(text)
    except (ValueError, TypeError):
        # ValueError for bad text, TypeError for None or a list.
        return default


def parse_float(text, default=None):
    """Convert text to a float, returning a default when it fails."""
    try:
        return float(text)
    except (ValueError, TypeError):
        return default


test_inputs = ["42", "3.14", "abc", "", None, "  7  ", [1]]

print("Input          parse_int    parse_float")
print("-" * 48)
for value in test_inputs:
    print(f"{repr(value):<14} {str(parse_int(value)):<12} {parse_float(value)}")

print("")
print("Catching TypeError as well as ValueError matters - None and")
print("lists raise TypeError, not ValueError.")

# Validating a range, not just a type.
def parse_age(text):
    """Parse an age, rejecting anything implausible."""
    value = parse_int(text)

    if value is None:
        return None, "not a number"
    if value < 0:
        return None, "cannot be negative"
    if value > 150:
        return None, "implausibly large"

    return value, "ok"


print("")
print("Validating as well as converting:")
for text in ["30", "-5", "200", "abc"]:
    result, message = parse_age(text)
    print(f"   {text:<6} -> {str(result):<6} ({message})")

## Converting between collections

The same explicit-conversion principle applies to containers.

In [ ]:
# Collection constructors take any iterable.
source = [3, 1, 2, 3, 1]

print("From a list", source)
print("   list()      ->", list(source))
print("   tuple()     ->", tuple(source))
print("   set()       ->", set(source), "<- duplicates removed, order lost")
print("   frozenset() ->", frozenset(source))

# Strings are iterables of characters.
text = "hello"
print("")
print("From the string 'hello':")
print("   list()  ->", list(text))
print("   set()   ->", set(text), "<- one 'l' only")
print("   tuple() ->", tuple(text))

# Building a dict needs pairs.
pairs = [("a", 1), ("b", 2)]
print("")
print("From pairs", pairs)
print("   dict() ->", dict(pairs))

# zip is the usual way to make those pairs.
keys = ["x", "y", "z"]
values = [10, 20, 30]
print("")
print("Zipping two lists into a dict:")
print("   dict(zip(keys, values)) ->", dict(zip(keys, values)))

# Joining back to a string needs str parts.
print("")
print("Back to a string:")
print("   ''.join(['a','b','c']) ->", "".join(["a", "b", "c"]))
try:
    "".join([1, 2, 3])
except TypeError as error:
    print("   ''.join([1,2,3])       ->", error)
print("   ''.join(str(n) for n in [1,2,3]) ->",
      "".join(str(number) for number in [1, 2, 3]))

## Converting to strings: `str()` vs `repr()`

Two different jobs, and the difference matters when debugging.

In [ ]:
from decimal import Decimal
import datetime

samples = [
    42,
    3.14,
    "hello",
    "  spaced  ",
    [1, 2],
    None,
    True,
    Decimal("1.10"),
    datetime.date(2026, 9, 13),
]

print("Value                str()                repr()")
print("-" * 64)
for value in samples:
    print(f"{str(type(value).__name__):<12} {str(value):<20} {repr(value)}")

print("")
print("str()  - readable, for end users")
print("repr() - unambiguous, for developers; ideally valid Python")
print("")
print("The difference is clearest for strings:")
text = "  hello  "
print("   print(text)       ->", text)
print("   print(repr(text)) ->", repr(text), "<- the spaces are visible")

print("")
print("This is why repr() is the better debugging tool. Chapter 27")
print("covers defining __str__ and __repr__ on your own classes.")

## Takeaways

1. Python is **strongly typed** — it promotes within the numeric tower
   (`int -> float -> complex`) but never between text and numbers.
2. `int()` **truncates towards zero**; `math.floor()` goes towards negative
   infinity. `int(-3.9)` is `-3`, `math.floor(-3.9)` is `-4`.
3. `int("42.7")` raises `ValueError` — convert through `float()` first.
4. Always wrap conversions of untrusted input in `try`/`except`, catching both
   `ValueError` and `TypeError`.
5. Validate **range** as well as type — a parsed number can still be nonsense.
6. Collection constructors accept any iterable; `set()` removes duplicates and
   loses order.
7. `str()` is for users, `repr()` is for developers — `repr()` shows quotes and
   whitespace.

## Try it yourself

1. Predict then check: `int(2.9)`, `int(-2.9)`, `round(2.5)`, `math.floor(-2.9)`.
2. Write `safe_divide(a, b)` handling non-numeric strings and division by zero.
3. Try `int()` on `"1_000"`, `"1e3"`, `" 42 "`, `"0b101"`. Which work?
4. Convert `[1, 2, 3]` to the string `"1-2-3"`.
5. Find a value where `str()` and `repr()` differ meaningfully.